# Getting started with pyfileindex

`pyfileindex` keeps a [pandas](https://pandas.pydata.org) `DataFrame` in sync with a directory tree: new files and directories, modified files, and deletions are all reflected after calling `update()`. The index is plain tabular data, so you can filter, group, or join it like any other DataFrame.

Install it with:
```shell
pip install pyfileindex
```
or
```shell
conda install -c conda-forge pyfileindex
```

This notebook covers:
1. Basic usage in the default **polling** mode (`watch=False`)
2. The same operations in **watch** mode (`watch=True`), and how it differs
3. Patterns for using `pyfileindex` inside your own project

## Setup

A couple of helpers used throughout this notebook: `touch()` to create/update a file like the Unix command, and `filter_function()` to restrict the index to `.txt` files.

In [ ]:
import os
from pyfileindex import PyFileIndex

In [ ]:
def touch(fname, times=None):
    with open(fname, "a"):
        os.utime(fname, times)

In [ ]:
def filter_function(file_name):
    return ".txt" in file_name

# 1. Polling mode (`watch=False`, the default)

By default `PyFileIndex` does not run anything in the background. Every call to `update()` rescans the directory tree, compares it against the previous index, and updates the DataFrame. This has no cost between calls, but the cost of `update()` itself grows with the size of the tree.

## Initialise PyFileIndex

In [ ]:
pfi = PyFileIndex(path=".", filter_function=filter_function, debug=True)
pfi

## Update PyFileIndex

In [ ]:
pfi.update()
pfi

## New directory

A new row appears with `is_directory=True`.

In [ ]:
os.makedirs("bla")

In [ ]:
pfi.update()
pfi

## New sub directory

Nested directories are picked up too -- `pyfileindex` scans recursively.

In [ ]:
os.makedirs("bla/bla")

In [ ]:
pfi.update()
pfi

## New file

The filtered `.txt` file shows up as a new row.

In [ ]:
touch("bla/bla/test.txt")

In [ ]:
pfi.update()
pfi

## Another new file

Files in different subdirectories are tracked independently.

In [ ]:
touch("bla/test.txt")

In [ ]:
pfi.update()
pfi

## Touch an existing file

Updating the modification time changes the `mtime` column for that row, without adding or removing rows.

In [ ]:
touch("bla/bla/test.txt", (1330712280, 1330712292))

In [ ]:
pfi.update()
pfi

## Remove a file

The corresponding row disappears from the index.

In [ ]:
os.remove("bla/bla/test.txt")

In [ ]:
pfi.update()
pfi

## Remove a directory

Removing a directory also drops its row; any files still inside would be removed from the index as well.

In [ ]:
os.rmdir("bla/bla")

In [ ]:
pfi.update()
pfi

## Clean up

In [ ]:
os.remove("bla/test.txt")

In [ ]:
os.rmdir("bla")

In [ ]:
pfi.update()
pfi

# 2. Watch mode (`watch=True`)

With `watch=True`, `PyFileIndex` starts a background thread (using [watchfiles](https://watchfiles.helpmanual.io)) that listens for file system events as they happen, instead of rescanning the tree on every `update()` call. This trades a small amount of background resource usage for much cheaper `update()` calls on large trees, since `update()` now just drains whatever change events have already been collected.

Two practical consequences:
- A change made *immediately before* calling `update()` may not have reached the background watcher yet. `update()` accepts a `timeout` argument (default 0.1s) to wait briefly for such pending changes before giving up and returning whatever is available.
- The background thread needs to be stopped explicitly with `close()`, or by using `PyFileIndex` as a context manager, once you're done with it. Forgetting to do so leaks a thread for as long as the process runs.

## Initialise PyFileIndex

In [ ]:
pfi = PyFileIndex(path=".", filter_function=filter_function, watch=True, debug=True)
pfi

## Update PyFileIndex

In [ ]:
pfi.update()
pfi

## New directory

The watcher reports this as soon as `update()` drains the pending change -- no rescan needed.

In [ ]:
os.makedirs("bla")

In [ ]:
pfi.update()
pfi

## New sub directory

Nested directories are reported the same way, via the background watcher.

In [ ]:
os.makedirs("bla/bla")

In [ ]:
pfi.update()
pfi

## New file

The filtered `.txt` file shows up as a new row once `update()` drains the event.

In [ ]:
touch("bla/bla/test.txt")

In [ ]:
pfi.update()
pfi

## Another new file

Files in different subdirectories are tracked independently.

In [ ]:
touch("bla/test.txt")

In [ ]:
pfi.update()
pfi

## Touch an existing file

The watcher reports a modification event, which updates the `mtime` column for that row.

In [ ]:
touch("bla/bla/test.txt", (1330712280, 1330712292))

In [ ]:
pfi.update()
pfi

## Remove a file

A deletion event removes the corresponding row from the index.

In [ ]:
os.remove("bla/bla/test.txt")

In [ ]:
pfi.update()
pfi

## Remove a directory

Deleting a directory removes its row; any files still inside are removed as well.

In [ ]:
os.rmdir("bla/bla")

In [ ]:
pfi.update()
pfi

## Clean up

In [ ]:
os.remove("bla/test.txt")

In [ ]:
os.rmdir("bla")

In [ ]:
pfi.update()
pfi

## Stop the background watcher

Once a `PyFileIndex` was created with `watch=True`, call `close()` to stop its background thread when you no longer need live updates.

In [ ]:
pfi.close()

# 3. Using pyfileindex in your own project

A few patterns that are useful once you embed `pyfileindex` in a larger application rather than calling it interactively.

## Scoping to a subdirectory with `open()`

`open()` returns a new `PyFileIndex` restricted to a subdirectory, reusing the parent's already-scanned data instead of rescanning from scratch.

In [ ]:
pfi = PyFileIndex(path=".", filter_function=filter_function)
os.makedirs("bla", exist_ok=True)
touch("bla/test.txt")
sub_pfi = pfi.open(path="bla")
sub_pfi

## Filtering for the files your application cares about

`filter_function` is called once per file (not per directory) and decides whether that file is kept in the index. Use it to limit the index to file types or naming patterns relevant to your application, which keeps the DataFrame small and `update()` fast.

In [ ]:
def only_python_files(file_name):
    return file_name.endswith(".py")


py_pfi = PyFileIndex(path=".", filter_function=only_python_files)
py_pfi

## Reliable cleanup with a context manager

If you use `watch=True` inside a long-running application, prefer the context manager form over manually calling `close()` -- it guarantees the background thread is stopped even if an exception is raised while the index is in use.

In [ ]:
with PyFileIndex(path=".", filter_function=filter_function, watch=True) as live_pfi:
    live_pfi.update()
    print(len(live_pfi), "files tracked")

In [ ]:
os.remove("bla/test.txt")
os.rmdir("bla")

# Next steps

See the [README](https://github.com/pyiron/pyfileindex#readme) for installation details and citation information, and the [source](https://github.com/pyiron/pyfileindex/blob/main/src/pyfileindex/pyfileindex.py) for the full API reference.